# Phase 1 — Data Audit

Run this notebook **before training**. All three visualizations are mandatory.

**Pass criteria:**
- VIZ 1.A: all class bars above red threshold line (≥25 instances)
- VIZ 1.B: boxes look tight and class labels are correct
- VIZ 1.C: no mass of boxes above 0.8 normalized width/height

In [ ]:
import os, sys, random, collections
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

CLASS_NAMES = ['arm', 'leg', 'torso', 'head', 'sensor']
COLORS = [(255,80,80), (80,200,80), (80,80,255), (255,200,0), (200,80,255)]

TRAIN_IMG_DIR = '../data/annotated/images/train'
TRAIN_LBL_DIR = '../data/annotated/labels/train'
VAL_IMG_DIR   = '../data/annotated/images/val'
TEST_IMG_DIR  = '../data/annotated/images/test'

RESULTS_DIR = '../results/figures'
os.makedirs(RESULTS_DIR, exist_ok=True)

## Dataset size check

In [ ]:
def count_images(directory):
    if not os.path.exists(directory):
        return 0
    return len([f for f in os.listdir(directory) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

n_train = count_images(TRAIN_IMG_DIR)
n_val   = count_images(VAL_IMG_DIR)
n_test  = count_images(TEST_IMG_DIR)
n_total = n_train + n_val + n_test

print(f'Train:  {n_train} images')
print(f'Val:    {n_val} images')
print(f'Test:   {n_test} images')
print(f'Total:  {n_total} images')
print()
if n_total < 300:
    print(f'⚠️  NEED {300-n_total} MORE IMAGES before proceeding.')
else:
    print('✅ Image count target met (≥300).')

## VIZ 1.A — Class distribution bar chart

In [ ]:
def count_class_distribution(labels_dir):
    counts = collections.Counter()
    if not os.path.exists(labels_dir):
        return counts
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith('.txt'):
            continue
        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                line = line.strip()
                if line:
                    class_id = int(line.split()[0])
                    counts[class_id] += 1
    return counts

train_counts = count_class_distribution(TRAIN_LBL_DIR)

fig, ax = plt.subplots(figsize=(8, 4))
bar_values = [train_counts[i] for i in range(5)]
bars = ax.bar(CLASS_NAMES, bar_values, color='steelblue', edgecolor='white')
ax.axhline(y=25, color='red', linestyle='--', label='min threshold (25)')
for bar, count in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(count), ha='center', va='bottom', fontsize=11)
ax.set_ylabel('Number of annotated instances (train set)')
ax.set_title('VIZ 1.A — Class distribution (training set)')
ax.legend()
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/viz1a_class_distribution.png', dpi=150)
plt.show()

print('⚠️  MANUAL CHECK: Any bar below the red line = not enough data for that class.')
print('   Fix before training.')

## VIZ 1.B — 20 random annotated images

In [ ]:
def draw_annotations(img_path, label_path, class_names):
    img = cv2.imread(img_path)
    if img is None:
        return np.zeros((224, 224, 3), dtype=np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    colors = [(255,80,80),(80,200,80),(80,80,255),(255,200,0),(200,80,255)]
    if os.path.exists(label_path):
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls, cx, cy, bw, bh = int(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                x1 = int((cx - bw/2) * w); y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w); y2 = int((cy + bh/2) * h)
                color = colors[cls % len(colors)]
                cv2.rectangle(img, (x1,y1), (x2,y2), color, 2)
                cv2.putText(img, class_names[cls], (x1, max(y1-6,0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)
    return img

img_dir = TRAIN_IMG_DIR
lbl_dir = TRAIN_LBL_DIR

if os.path.exists(img_dir):
    img_files = [f for f in os.listdir(img_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))]
    sample = random.sample(img_files, min(20, len(img_files)))

    n_cols = 5
    n_rows = max(1, (len(sample) + n_cols - 1) // n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, n_rows * 4))
    if n_rows == 1:
        axes = [axes]
    axes_flat = [ax for row in axes for ax in (row if hasattr(row, '__iter__') else [row])]

    for ax, fname in zip(axes_flat, sample):
        stem = os.path.splitext(fname)[0]
        lbl_path = os.path.join(lbl_dir, stem + '.txt')
        img_path = os.path.join(img_dir, fname)
        img = draw_annotations(img_path, lbl_path, CLASS_NAMES)
        ax.imshow(img)
        ax.set_title(fname[:20], fontsize=8)
        ax.axis('off')

    for ax in axes_flat[len(sample):]:
        ax.axis('off')

    plt.suptitle('VIZ 1.B — 20 random annotated training images', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/viz1b_annotation_sample.png', dpi=120)
    plt.show()
    print('⚠️  MANUAL CHECK: Are boxes tight? Are class labels correct?')
    print('   Common errors: wrong class ID, box covering whole image, missing annotations.')
else:
    print(f'No images yet in {img_dir}. Run after dataset is populated.')

## VIZ 1.C — Bounding box size distribution

In [ ]:
widths, heights = [], []
if os.path.exists(TRAIN_LBL_DIR):
    for label_file in os.listdir(TRAIN_LBL_DIR):
        if not label_file.endswith('.txt'): continue
        with open(os.path.join(TRAIN_LBL_DIR, label_file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    widths.append(float(parts[3]))
                    heights.append(float(parts[4]))

if widths:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(widths, bins=30, color='steelblue', edgecolor='white')
    axes[0].set_xlabel('Normalized box width'); axes[0].set_ylabel('Count')
    axes[0].set_title('Box width distribution')
    axes[0].axvline(x=0.8, color='red', linestyle='--', label='suspiciously large (>0.8)')
    axes[0].legend()

    axes[1].hist(heights, bins=30, color='darkorange', edgecolor='white')
    axes[1].set_xlabel('Normalized box height'); axes[1].set_ylabel('Count')
    axes[1].set_title('Box height distribution')
    axes[1].axvline(x=0.8, color='red', linestyle='--', label='suspiciously large (>0.8)')
    axes[1].legend()

    plt.suptitle('VIZ 1.C — Bounding box size distribution')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/viz1c_box_sizes.png', dpi=150)
    plt.show()
    print(f'Total annotations: {len(widths)}')
    print(f'Boxes with width > 0.8:  {sum(1 for w in widths if w > 0.8)}')
    print(f'Boxes with height > 0.8: {sum(1 for h in heights if h > 0.8)}')
    print(f'Boxes with width < 0.02:  {sum(1 for w in widths if w < 0.02)} (too small)')
    print()
    print('⚠️  MANUAL CHECK: Boxes >0.8 are likely annotation errors. Boxes <0.02 are too small.')
else:
    print('No labels found yet.')

## Annotation coverage check

In [ ]:
def check_annotation_coverage(img_dir, lbl_dir):
    if not os.path.exists(img_dir):
        print(f'Directory not found: {img_dir}')
        return
    img_files = {os.path.splitext(f)[0] for f in os.listdir(img_dir)
                 if f.lower().endswith(('.jpg','.jpeg','.png'))}
    lbl_files = {os.path.splitext(f)[0] for f in os.listdir(lbl_dir)
                 if f.endswith('.txt')} if os.path.exists(lbl_dir) else set()

    missing_labels = img_files - lbl_files
    empty_labels = set()
    for stem in lbl_files:
        lbl_path = os.path.join(lbl_dir, stem + '.txt')
        if os.path.getsize(lbl_path) == 0:
            empty_labels.add(stem)

    print(f'Images:         {len(img_files)}')
    print(f'Labels:         {len(lbl_files)}')
    print(f'Missing labels: {len(missing_labels)}')
    print(f'Empty labels:   {len(empty_labels)}')
    if missing_labels:
        print('Images missing labels:', list(missing_labels)[:5], '...' if len(missing_labels) > 5 else '')
    if not missing_labels and not empty_labels:
        print('✅ 100% annotation coverage.')

print('=== Train set ===')
check_annotation_coverage(TRAIN_IMG_DIR, TRAIN_LBL_DIR)

## Phase 1 completion checklist

- [ ] ≥ 300 images collected
- [ ] All images annotated in YOLO format
- [ ] Train/val/test/calibration split created
- [ ] VIZ 1.A saved — all class bars above red threshold
- [ ] VIZ 1.B saved — 20 images visually inspected, boxes correct
- [ ] VIZ 1.C saved — box size distribution looks reasonable
- [ ] 100% annotation coverage confirmed